# Less is More: Encoder-only Audio-Visual Segmentation

Ilpo Viertola, Vladimir Iashin, Sophie Tötterström, Esa Rahtu  
*Tampere University*

## 0. Environment Setup

```bash
conda create --name ease python=3.12 -y
conda activate ease
# Select torch according to your system specs.
pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu126
pip install -e .
```

In [ ]:
from pathlib import Path

import torch
from PIL import Image
import torch.nn.functional as F
from torchvision import tv_tensors
from torchvision.io import read_video
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.v2 import functional as Fv

from models.ease_vit import EASEViT
from models.encoders.vit import ViT
from models.vggish.vggish import VGGish
from models.fusers.av_aligner import AVAligner
from utils.save_mask_util import colored_masks
from models.feat_upscalers.conv_scaler import ConvScaler
from models.query_generators.attention_generator import AttentionGenerator
from models.query_generators.aud_enhancer_w_clustering import ClusteringTokenFeatureEnhancer

CHECKPOINT_DIR = Path("./assets/ease-checkpoints")
CP_DOWNLOAD_URL = "https://a3s.fi/swift/v1/ease-public/ease-vit-l-384-ss.pt"
VIDEO_DIR = Path("./assets/videos")
SAVE_DIR = Path("./assets/results")

VGGISH_URLS = {
    "vggish": "https://github.com/harritaylor/torchvggish/releases/download/v0.1/vggish-10086976.pth",
    "pca": "https://github.com/harritaylor/torchvggish/releases/download/v0.1/vggish_pca_params-970ea276.pth",
}


## 1. Model Setup

In [ ]:
network = EASEViT(
    num_classes=71,
    num_q=100,
    fusion_at_blocks=[20, 21, 22, 23],
    late_fusion=True,
    encoder=ViT(
        img_size=(384, 384),
        backbone_name="facebook/dinov3-vitl16-pretrain-lvd1689m"
    ),
    query_generator=AttentionGenerator(
        query_num=100,
        num_layers=2,
        project_output_dim=1024
    ),
    feat_upscaler=ConvScaler(
        encoder_patch_size=(16, 16),
        embed_dim=1024
    ),
    audio_enhance=ClusteringTokenFeatureEnhancer(
        embed_dim=320,
        k=71,
        aud_dim=128
    ),
    fuser=AVAligner(
        aud_dim=320,
        dim=1024,
        num_heads=8,
        num_clusters=5
    )
)

In [ ]:
if not CHECKPOINT_DIR.exists():
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

cp_name = CP_DOWNLOAD_URL.split("/")[-1]
cp_path = CHECKPOINT_DIR / cp_name
if not cp_path.exists():
    import requests
    print(f"Downloading checkpoint from {CP_DOWNLOAD_URL} to {cp_path}")
    r = requests.get(CP_DOWNLOAD_URL, allow_redirects=True)
    open(cp_path, 'wb').write(r.content)

In [ ]:
ckpt = torch.load(cp_path, map_location="cpu")
ckpt = {k.replace("network.", ""): v for k, v in ckpt.items() if k != "criterion.empty_weight"}
network.load_state_dict(ckpt)

In [ ]:
vggish = VGGish(urls=VGGISH_URLS, postprocess=False)

## 2. Data Setup

In [ ]:
class VideoAudioDataset(Dataset):
    def __init__(self, videos, frame_size=(384, 384)):
        self.videos = [Path(video) for video in videos]
        self.frame_size = frame_size

    def __len__(self):
        return len(self.videos)

    def __getitem__(self, index):
        video_path = self.videos[index]

        frames, audio, info = read_video(
            str(video_path),
            pts_unit="sec",
        )

        # read_video returns frames as [T, H, W, C].
        # Convert to [T, C, H, W] and sample at 1 FPS.
        frames = frames.permute(0, 3, 1, 2)
        fps = float(info["video_fps"])

        sample_count = max(1, int(torch.ceil(torch.tensor(len(frames) / fps))))
        frame_indices = (
            torch.arange(sample_count, dtype=torch.float32) * fps
        ).round().long()
        frame_indices = frame_indices.clamp(max=len(frames) - 1).unique()

        frames = frames[frame_indices]
        frames = Fv.resize(frames, self.frame_size)
        frames = tv_tensors.Video(frames)

        # Audio shape: [samples, channels].
        audio = audio.float()
        audio = audio.transpose(0, 1)
        sample_rate = int(info["audio_fps"])

        return frames, audio, sample_rate

In [ ]:
videos = list(VIDEO_DIR.glob("*.mp4"))
assert len(videos) > 0, f"No videos found in {VIDEO_DIR}. Please add some .mp4 files to the directory."

dataset = VideoAudioDataset(videos)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

## 3. Run Inference

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
network.to(device)
network.eval()
vggish.to(device)
vggish.eval()

j = 0

with torch.inference_mode() and torch.autocast(device_type=device.type, dtype=torch.float16):
    for frames, audio, sample_rate in dataloader:
        frames = frames.to(device)
        B, T, C, H, W = frames.shape
        assert B == 1, "Batch size must be 1 for demo implementation."

        # Process audio through VGGish to get embeddings.
        audio_embeddings = vggish(audio[0].cpu().numpy(), sample_rate[0].item())
        audio_embeddings = audio_embeddings.view(B * T, 1, -1)
        audio_embeddings = audio_embeddings.to(frames.device)

        # Run the EASE model.
        frames = frames.view(B * T, C, H, W)
        frames = frames / 255.0
        mask_logits_per_layer, class_logits_per_layer, _ = network(frames, audio_embeddings)

        mask_logits = mask_logits_per_layer[-1]
        mask_logits = F.interpolate(mask_logits, 224, mode="bilinear")
        class_logits = class_logits_per_layer[-1]
        logits = torch.einsum(
            "bqhw, bqc -> bchw",
            mask_logits.sigmoid(),
            class_logits.softmax(dim=-1)[..., :-1],
        )

        for i, logit in enumerate(logits):
            pred = torch.argmax(logit, dim=0)[None, ...]
            SAVE_DIR.mkdir(parents=True, exist_ok=True)
            colored_pred = colored_masks(pred.cpu().numpy())[0]
            im = Image.fromarray(colored_pred).resize((360, 240))
            im.save(SAVE_DIR / f"pred_{j}_{i}.png")
        j += 1
